<!--
---
PURPOSE: Discover videos and align frame times to NWB timebase.
REQUIRES:
  - outputs/reports/session_inventory.parquet
PRODUCES:
  - outputs/video/session_{id}_video_manifest.json
  - outputs/video/session_{id}_frame_times.parquet
  - outputs/video/session_{id}_preview.mp4
WHAT_NEXT: notebooks/06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb
---
-->

# 05 Video I/O and Frame Timebase

**Purpose**
Discover videos and align frame times to NWB timebase.

**Requires**
- `outputs/reports/session_inventory.parquet`

**Produces**
- `outputs/video/session_{id}_video_manifest.json`
- `outputs/video/session_{id}_frame_times.parquet`
- `outputs/video/session_{id}_preview.mp4`

**What to run next**
- `notebooks/06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb`

We locate videos, compute per-frame timestamps, and validate alignment.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "05_Video_IO_and_Frame_Timebase.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Build video manifest and frame times
This step enforces the timebase alignment tiers and produces QC flags.

In [ ]:
from io_sessions import load_sessions_csv, get_session_bundle
from viz import plot_video_alignment

sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    manifest = bundle.load_video_manifest(prefer_download=False)
    frame_times = bundle.load_frame_times()
    print(manifest.get("alignment_method"), manifest.get("qc_flags"))
    plot_video_alignment(frame_times)